In [1]:
import logging
# Configuration du logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

In [2]:
import HBPy
from HBPy.Molecule.Crystal import Crystal,Atom

/home/bulou/venv/ATOMOD/lib/python3.12/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


In [3]:
status={
        'abtem':True,
        'feff':True,
        'atomic probability map':True,
        'optimization':False,
    }
config={
       'root_dir':'simul',
          'train':{
            'TEM_img_dir':"train/images",  # répertoire de stockage des images TEM
            'prob_maps_img_dir':"train/prob_maps"  # répertoire de stockage des images TEM
        },
 
    'structure':{
         'optimization':status['optimization'],
    'composition':['Pt','Co','Au','Ir','Pd'],
    'radius':5.5,
    'a':3.9,
    },
     'nvaccum':2.0,
    'image':{
     'xmin':0.0,
     'xmax':0.0,
     'ymin':0.0,
     'ymax':0.0
    },
   'atomic presence probability map':{
            'ninter':{ # nombre d'intervalles entre deux positions atomiques
                'x':20,
                'y':20,
                'z':2
            },
         'sigma': .6  # en Å, largeur de la gaussienne ~ rayon atomique ou un peu moins
   },
            'atomic probability map':{
            'status':status['atomic probability map']
        },
           'abtem':{
            'status':status['abtem'],
            'dx':0.04,
            'dy':0.04,
            'dz':4.08/2,
            'energy':300e3,
            'focal spread':40,
            'semiangle cutoff':20,
            'defocus':200,
            'cell scale':1.1
            },
 }

_______________________________________
# Etape 1 : construire la nanoparticules
 _______________________________________
#   Etape 1.1 : la structure

In [4]:
NP=Crystal()
NP.build(a=config['structure']['a'],
    radius=config['structure']['radius'],
    materials='NP')
NP.origin_at_mass_center()
logger.info(f"min={NP.qmin} max={NP.qmax}")
logger.info(f"Mass center={NP.MC}")
logger.info(f"Number of atoms={len(NP.atoms)}")

2026-06-17 15:44:34,127 [INFO] - <module>() - min=[-4.875 -4.875 -4.875] max=[4.875 4.875 4.875]
2026-06-17 15:44:34,129 [INFO] - <module>() - Mass center=[-1.31208176e-16 -1.13040890e-15 -2.76546462e-15]
2026-06-17 15:44:34,130 [INFO] - <module>() - Number of atoms=44


 #   Etape 1.2 : la distribution chimique   

In [5]:
NP.set_composition(config['structure']['composition'])
NP.save(prefix="NP",fmt='xyz',directory="./")

#   Etape 1.3 : (optionnelle) l'optimisation structurale et/ou chimique

In [6]:
 if config['structure']['optimization']:
     NP.optimize_ase()
     logger.info(f"Optimization DONE!")

In [7]:
import py3Dmol
from pathlib import Path

In [8]:
chemin_fichier = Path("NP.xyz")
xyz_data = chemin_fichier.read_text(encoding="utf-8")
vue = py3Dmol.view(width=400, height=400)
vue.addModel(xyz_data, "xyz")
vue.setStyle({'sphere': {'colorscheme': 'Jmol', 'scale': 1.0}, 'spacefill': {}})
vue.zoomTo()
vue.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.